# VerSe 3D to 2D Processing Pipeline (Generalized)
This notebook serves as an educational guide and a functional pipeline for converting 3D VerSe (Vertebrae Segmentation) CT scans into a 2D dataset.

### The Goal:
- Load 3D NIfTI volumes.
- Apply clinical bone windowing (Hounsfield Units).
- Extract 2D sagittal slices that contain significant vertebral structures.
- Generate **Semantic**, **Instance**, and **Panoptic** annotations.

### Supported Styles:
- **`ade20k`**: Standard folder structure (`images/`, `annotations_semantic/`, `ade20k_panoptic/`).
- **`cityscapes`**: Uses suffix naming (`_leftImg8bit.png`, `_gtFine_labelIds.png`) and Instance ID encoding (`CatID * 1000 + InstID`).
- **`mapillary`**: Uses `labels/`, `instances/`, and `panoptic/` folders.

**Note:** Each style is isolated in its own folder within the output directory (e.g., `dataset_verse_2d/ade20k/...`).


In [ ]:
import os
import glob
import json
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from data_utilities import *

# --- CONFIGURATION PANEL ---
# Adjust these variables to control the processing logic.
INPUT_ROOT = '../data'
OUTPUT_ROOT = '../dataset_verse_2d'
SPLITS = ['train', 'val', 'test']

# Style: 'ade20k', 'cityscapes', or 'mapillary'
EXPORT_STYLE = 'ade20k' 

# Tasks to generate: ['semantic', 'instance', 'panoptic']
SELECTED_TASKS = ['semantic', 'instance', 'panoptic']

# Filtering parameters
THRESHOLD_PERCENT = 0.20  # Keep slices with area > 20% of the maximum slice area in that volume
MIN_PIXELS = 500          # Absolute minimum pixels to keep a slice
FORCE_REPROCESS = True

V_NAMES = ['C1','C2','C3','C4','C5','C6','C7','T1','T2','T3','T4','T5','T6','T7','T8','T9','T10','T11','T12','L1','L2','L3','L4','L5','L6','Sacrum','Cocc','T13']

def map_semantic_class(inst_id):
    """
    Generalized mapping from Instance IDs to Semantic Classes.
    VerSe Default: 1=Cervical, 2=Thoracic, 3=Lumbar, 0=Background
    """
    if 1 <= inst_id <= 7: return 1  # Cervical
    if (8 <= inst_id <= 19) or (inst_id == 28): return 2  # Thoracic
    if 20 <= inst_id <= 25: return 3  # Lumbar
    return 0 # Background


## 1. Anatomy of a 3D Subject
Medical imaging (CT/MRI) is stored in 3D volumes, usually in the **NIfTI** (`.nii.gz`) format. A single 3D volume is essentially a stack of hundreds of 2D slices.


In [ ]:
# Load a sample subject for exploration
sample_sub = 'sub-verse500'
sample_split = 'train'
sample_raw_path = glob.glob(os.path.join(INPUT_ROOT, sample_split, 'rawdata', sample_sub, '*_ct.nii.gz'))[0]
sample_msk_path = glob.glob(os.path.join(INPUT_ROOT, sample_split, 'derivatives', sample_sub, '*_seg-vert_msk.nii.gz'))[0]

img_nii = nib.load(sample_raw_path)
msk_nii = nib.load(sample_msk_path)

print(f"Subject: {sample_sub}")
print(f"3D Image Shape: {img_nii.shape}")
print(f"3D Mask Shape:  {msk_nii.shape}")
print(f"Voxel Spacing:  {img_nii.header.get_zooms()}")


## 2. Bone Windowing (Hounsfield Units)
Raw CT values are in Hounsfield Units (HU). Bones typically have high HU (200 to 1000+), while soft tissue is lower. 
To help the model see the spine clearly, we "window" the image to focus only on the range relevant to bone.


In [ ]:
def window_image(img, min_hu=-500, max_hu=1300):
    """Apply bone windowing to CT images."""
    img = np.clip(img, min_hu, max_hu)
    img = (img - min_hu) / (max_hu - min_hu) * 255.0
    return img.astype(np.uint8)

# Extract a middle slice for visualization
middle_idx = img_nii.shape[2] // 2
raw_data = img_nii.get_fdata()[:, :, middle_idx]

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(raw_data, cmap='gray')
ax[0].set_title("Raw CT Slice (Low Contrast)")
ax[1].imshow(window_image(raw_data), cmap='gray')
ax[1].set_title("Bone Windowed Slice (Clear Spine)")
plt.show()


## 3. Slice Selection Strategy
Not every 2D slice in a 3D volume is useful. Slices at the very edges of the scan might be empty or contain only tiny bone fragments. 
We use a **Dynamic Threshold** to keep only high-quality slices.


In [ ]:
# Resample and Reorient first (Standardizing space)
im_iso = resample_nib(img_nii, (1,1,1), 3)
ms_iso = resample_nib(msk_nii, (1,1,1), 0)
im_np = reorient_to(im_iso, ('I','P','L')).get_fdata()
ms_np = reorient_to(ms_iso, ('I','P','L')).get_fdata()

# Calculate area of mask in each slice
areas = [np.sum(ms_np[:,:,i] > 0) for i in range(im_np.shape[2])]
max_area = max(areas)
threshold = max(MIN_PIXELS, THRESHOLD_PERCENT * max_area)

plt.figure(figsize=(10, 4))
plt.plot(areas, label='Vertebrae Pixels')
plt.axhline(y=threshold, color='r', linestyle='--', label=f'Threshold ({int(threshold)}px)')
plt.title("Bone Area per Slice (Sagittal View)")
plt.xlabel("Slice Index")
plt.ylabel("Number of Pixels")
plt.legend()
plt.show()

kept_indices = [i for i, a in enumerate(areas) if a >= threshold]
removed_indices = [i for i, a in enumerate(areas) if a < threshold and a > 0]

if kept_indices and removed_indices:
    fig, ax = plt.subplots(1, 2, figsize=(12, 6))
    ax[0].imshow(window_image(im_np[:,:,kept_indices[len(kept_indices)//2]]), cmap='gray')
    ax[0].set_title(f"KEPT: Slice {kept_indices[len(kept_indices)//2]}")
    ax[1].imshow(window_image(im_np[:,:,removed_indices[0]]), cmap='gray')
    ax[1].set_title(f"REMOVED: Slice {removed_indices[0]}")
    plt.show()


## 4. Demystifying the Annotations
We generate three types of ground truth:
1. **Semantic:** Groups vertebrae into regions (Cervical, Thoracic, Lumbar).
2. **Instance:** Each individual vertebra has its own unique ID.
3. **Panoptic:** Encodes Instance IDs into an RGB image for panoptic segmentation tasks.


In [ ]:
def get_panoptic_encoding(inst_msk):
    unique_ids = np.unique(inst_msk).astype(int)
    unique_ids = unique_ids[unique_ids > 0]
    pan_img = np.zeros((inst_msk.shape[0], inst_msk.shape[1], 3), dtype=np.uint8)
    segments_info = []
    for uid in unique_ids:
        mask = (inst_msk == uid)
        area = int(np.sum(mask))
        pan_img[mask, 0] = uid % 256
        pan_img[mask, 1] = uid // 256
        pan_img[mask, 2] = 0
        y, x = np.where(mask)
        bbox = [float(np.min(x)), float(np.min(y)), float(np.max(x)-np.min(x)), float(np.max(y)-np.min(y))]
        segments_info.append({
            'id': int(uid), 'category_id': int(uid), 'isthing': True, 'iscrowd': 0, 'area': area, 'bbox': bbox, 'bbox_mode': 1
        })
    return pan_img, segments_info

# Choose one slice to visualize all types
vis_idx = kept_indices[len(kept_indices)//2]
img_slice = window_image(im_np[:,:,vis_idx])
inst_slice = ms_np[:,:,vis_idx].astype(np.uint8)

# Create Semantic using the generalized mapping from Configuration Panel
sem_slice = np.vectorize(map_semantic_class)(inst_slice).astype(np.uint8)

pan_slice, _ = get_panoptic_encoding(inst_slice)

fig, ax = plt.subplots(1, 4, figsize=(20, 5))
ax[0].imshow(img_slice, cmap='gray'); ax[0].set_title("Image")
ax[1].imshow(sem_slice, cmap='tab10'); ax[1].set_title("Semantic (Regions)")
ax[2].imshow(inst_slice, cmap='nipy_spectral'); ax[2].set_title("Instance (Vertebrae)")
ax[3].imshow(pan_slice); ax[3].set_title("Panoptic (RGB Encoding)")
plt.show()


## 5. The Full Processing Pipeline (Style Mapping)
This section handles the complex logic of mapping folder structures and ID encodings based on your chosen `EXPORT_STYLE`. Each style is saved in its own sub-directory to prevent data overwriting.


In [ ]:
def get_paths_for_style(style, subject_id, split_name, base_output):
    """Returns the directory mapping and file naming for the selected style."""
    style_root = os.path.join(base_output, style)
    paths = {
        'img': os.path.join(style_root, split_name, 'images'),
        'sem': os.path.join(style_root, split_name, 'annotations_semantic'),
        'ins': os.path.join(style_root, split_name, 'annotations_instance'),
        'pan': os.path.join(style_root, split_name, f'{style}_panoptic'),
        'suffixes': {'img': '', 'sem': '', 'ins': '', 'pan': ''}
    }
    
    if style == 'cityscapes':
        paths['img'] = os.path.join(style_root, split_name, 'leftImg8bit', subject_id)
        paths['sem'] = os.path.join(style_root, split_name, 'gtFine', subject_id)
        paths['ins'] = os.path.join(style_root, split_name, 'gtFine', subject_id)
        paths['pan'] = os.path.join(style_root, split_name, 'cityscapes_panoptic')
        paths['suffixes'] = {'img': '_leftImg8bit', 'sem': '_gtFine_labelIds', 'ins': '_gtFine_instanceIds', 'pan': ''}
    
    elif style == 'mapillary':
        paths['sem'] = os.path.join(style_root, split_name, 'labels')
        paths['ins'] = os.path.join(style_root, split_name, 'instances')
        paths['pan'] = os.path.join(style_root, split_name, 'panoptic')
        
    return paths

def export_granular(sub_id, s_idx, img_win, inst_msk, sem_msk, style, split_name, base_output):
    # 0. Get Style Config
    cfg = get_paths_for_style(style, sub_id, split_name, base_output)
    base_name = f"{sub_id}_sag_{s_idx:03d}"
    
    # Panoptic Encoding
    pan_img, segments = get_panoptic_encoding(inst_msk)
    sub_num = int(''.join(filter(str.isdigit, sub_id)))
    image_id = sub_num * 1000 + s_idx
    info = {"file_name": f"{base_name}{cfg['suffixes']['img']}.png", "id": image_id, "segments": segments, "width": img_win.shape[1], "height": img_win.shape[0]}

    # 1. Save Image
    os.makedirs(cfg['img'], exist_ok=True)
    Image.fromarray(img_win).save(os.path.join(cfg['img'], f"{base_name}{cfg['suffixes']['img']}.png"))

    # 2. Save Semantic
    if 'semantic' in SELECTED_TASKS:
        os.makedirs(cfg['sem'], exist_ok=True)
        Image.fromarray(sem_msk).save(os.path.join(cfg['sem'], f"{base_name}{cfg['suffixes']['sem']}.png"))

    # 3. Save Instance
    if 'instance' in SELECTED_TASKS:
        os.makedirs(cfg['ins'], exist_ok=True)
        save_ins = inst_msk.copy()
        if style == 'cityscapes':
            # Cityscapes ID Encoding: CatID * 1000 + InstID
            save_ins = (save_ins.astype(np.uint32) * 1000 + 1)
            save_ins[inst_msk == 0] = 0
            
        Image.fromarray(save_ins).save(os.path.join(cfg['ins'], f"{base_name}{cfg['suffixes']['ins']}.png"))

    # 4. Save Panoptic
    if 'panoptic' in SELECTED_TASKS:
        os.makedirs(cfg['pan'], exist_ok=True)
        Image.fromarray(pan_img).save(os.path.join(cfg['pan'], f"{base_name}{cfg['suffixes']['pan']}.png"))

    return info

def process_subject(sub_id, raw_path, msk_path, split_name, style, base_output):
    try:
        im_iso, ms_iso = resample_nib(nib.load(raw_path),(1,1,1),3), resample_nib(nib.load(msk_path),(1,1,1),0)
        im_np, ms_np = reorient_to(im_iso,('I','P','L')).get_fdata(), reorient_to(ms_iso,('I','P','L')).get_fdata()
        areas = [np.sum(ms_np[:,:,i]>0) for i in range(im_np.shape[2])]
        max_area = max(areas)
        kept = [i for i, a in enumerate(areas) if a >= max(MIN_PIXELS, THRESHOLD_PERCENT * max_area)]
        
        all_info = []
        for i in kept:
            img_win = window_image(im_np[:,:,i])
            inst_msk = ms_np[:,:,i].astype(np.uint8)
            sem_msk = np.vectorize(map_semantic_class)(inst_msk).astype(np.uint8)
            info = export_granular(sub_id, i, img_win, inst_msk, sem_msk, style, split_name, base_output)
            all_info.append(info)
        return all_info
    except Exception as e:
        return []

categories = [{'id': i+1, 'name': V_NAMES[i], 'isthing': 1} for i in range(len(V_NAMES))]
categories.append({'id': 0, 'name': 'background', 'isthing': 0})

all_processed_data = {}

for split in SPLITS:
    out_n = split
    style_split_root = os.path.join(OUTPUT_ROOT, EXPORT_STYLE, out_n)
    print(f'\nProcessing Split: {split} (Style: {EXPORT_STYLE})')
    os.makedirs(style_split_root, exist_ok=True)
    
    subs = sorted(glob.glob(os.path.join(INPUT_ROOT, split, 'rawdata', 'sub-*')))
    all_res = []
    for s in tqdm(subs, desc=split):
        sid = os.path.basename(s)
        raw = glob.glob(s+'/*_ct.nii.gz')[0]
        msk_l = glob.glob(os.path.join(INPUT_ROOT, split, 'derivatives', sid, '*_seg-vert_msk.nii.gz'))
        if msk_l:
            all_res.extend(process_subject(sid, raw, msk_l[0], out_n, EXPORT_STYLE, OUTPUT_ROOT))
    
    # Save Metadata JSON (COCO-style) inside the style-specific split folder
    json_p = os.path.join(style_split_root, f'verse_{out_n}_metadata.json')\n    images = [{"id": r["id"], "file_name": r["file_name"], "width": r["width"], "height": r["height"]} for r in all_res]
    annotations = [{"image_id": r["id"], "file_name": r["file_name"], "segments_info": r["segments"]} for r in all_res]
    
    with open(json_p, 'w') as f:
        json.dump({"images": images, "annotations": annotations, "categories": categories}, f, indent=4)
    
    all_processed_data[out_n] = all_res


## 6. Final Dataset Statistics
Let's verify what we've generated and check for any class imbalances.


In [ ]:
# Summary Table
print(f"{'Split':<10} | {'Total Slices':<15}")
print("-" * 30)
for split, data in all_processed_data.items():
    print(f"{split:<10} | {len(data):<15}")

# Class Distribution (Frequency of each vertebra ID)
if all_processed_data.get('train'):
    all_cat_ids = []
    for img in all_processed_data['train']:
        for seg in img['segments']:
            all_cat_ids.append(seg['category_id'])
    
    counts = {i+1: all_cat_ids.count(i+1) for i in range(len(V_NAMES))}
    
    plt.figure(figsize=(15, 5))
    plt.bar([V_NAMES[i-1] for i in counts.keys()], counts.values())
    plt.xticks(rotation=45)
    plt.title("Class Distribution in Training Set (Vertebra Occurrences)")
    plt.ylabel("Number of Slices")
    plt.show()
